In [12]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

# auto-reload library if developing library functionalities
%reload_ext autoreload
%autoreload 2
# Add the old-can-decoder-c56d4fe6 directory to sys.path
sys.path.append(os.path.abspath('old-can-decoder-c56d4fe6'))

# import library
from PandaCANDecoder.decoder import Decoder as OldDecoder
from can_decoder.decoder import Decoder
from can_decoder.message import Message
from can_decoder.signal import Signal
import xml.etree.ElementTree as ET


In [23]:

file_path = './data/2025 Jeep Grand Cherokee 4xE.gmd'
tree = ET.parse(file_path)
root = tree.getroot()


# Group in a dict ECU's by network name
target_network = 'HS CAN5' 


ecus = root.find('GMLANECUS').findall('GMLANECU')
ecus_by_network = {}
for ecu in ecus:
    network_name = ecu.find('NetworkName').text
    if network_name not in ecus_by_network:
        ecus_by_network[network_name] = []
    ecus_by_network[network_name].append(ecu)

network_messages = []
network_signals = []
for ecu in ecus_by_network[target_network]:
    gmlandids_elem = ecu.find("GMLANDIDs")
    messages = gmlandids_elem.findall("GMLANDID") if gmlandids_elem is not None else []
    # This step removes ECU's, "flattening" the info into network with messages 
    network_messages.extend(messages)
    for msg in messages:
        signals_elem = msg.find("GMLANIDSignals")
        signals = signals_elem.findall("Signal") if signals_elem is not None else []
        network_signals.extend(signals)
# Dictionary : keys are signal names, values are dict with keys "factor" and "offset"
signal_transforms = {
    s.find("Description").text: {
        "factor": float(s.find("Mult").text) if s.find("Mult") is not None else 1.0,
        "offset": float(s.find("Add").text) if s.find("Add") is not None else 0.0
    } for s in network_signals
}
signal_transforms

{'HVAC_Coolant_Temp__C': {'factor': 1.0, 'offset': -40.0},
 'HVAC_LVBattery_Voltage__V': {'factor': 0.2, 'offset': -3.6},
 'HVAC_External_Temp__C': {'factor': 0.5, 'offset': -40.0},
 'HVAC_External_Temp_Filtered__C': {'factor': 0.5, 'offset': -40.0},
 'HVAC_Cabin_Temp__C': {'factor': 0.1, 'offset': -50.0},
 'HVAC_Evaporator_Temp__C': {'factor': 0.1, 'offset': -50.0},
 'HVAC_Compressor_Solenoid_DutyCycle__%': {'factor': 0.1, 'offset': 0.0},
 'HVAC_Compressor_Solenoid_Current__A': {'factor': 0.01, 'offset': 0.0},
 'HVAC_Blower_Motor_DutyCycle__%': {'factor': 0.1, 'offset': 0.0},
 'HVAC_Front_Blower_Fan_Speed': {'factor': 0.5, 'offset': 0.0},
 'HVAC_Engine_Spd__RPM': {'factor': 0.25, 'offset': 0.0},
 'HVAC_Vehicle_Spd_LowRes__kph': {'factor': 1.0, 'offset': 0.0}}

In [21]:
ecus_by_network[target_network][2].find("GMLANDIDs").findall("GMLANDID")

AttributeError: 'NoneType' object has no attribute 'findall'

In [4]:
print("\n".join(s.find("Description").text for s in network_signals))

In [ ]:
message = network_messages[0]
# f"{int(network_messages[3].find('ID').text):X}" 
# or hex(int(network_messages[3].find("ID").text))[2:].upper()

'616'

'18DAF110'

In [ ]:
m = Message(

    msg_id=ecu.find("USDTResponseID").text,
    name=f"{int(message.find('ID').text):X}_{message.find('Description').text}",
    signals=[
        Signal(
            name=s.find("Name").text,
            start_bit=int(s.find("StartBit").text),
            bit_length=int(s.find("BitLength").text),
            is_signed=s.find("IsSigned").text == "true",
            factor=float(s.find("Factor").text),
            offset=float(s.find("Offset").text)
        )
        for s in network_messages[0].find("Signals").findall("Signal")
    ]
)

In [ ]:
## List of messages for the decoder

for message in network_messages:
    print(message.find("Description").text)

Components Miscelaneous 1
Torque Converter Slip
State of Torque Converter Clutch
RPMs and Temps Data
ActualCrankshaft Torque
Lateral and Longitudinal Accelerations
External Temperature
Break Data and others
ABS ESC and Brake Data
Wheelspeed Data
Bake Pedal Position
Transfer Case Range Status
Engine Gas Pedal and Others
Atmospheric Pressure and Thermal Management
Engine Oil Temperature
Terrain Mode and Engine Water Temperature
Torque Sums Max Mins
Electrical Machine Torque Data
AWD System Status
Current and Target Gear
Slope
Steering Angle
Engine Oil Temp etc MODE 21
Accum Air Port Mass Flow
Barometric Pressure
Engine Coolant Temperature
Shut Down Engine Coolant Temperature
Intake Air Temperature
Ambient Temperature
CAT Modeled Temperature
RPM vs Vehicle Speed Ratio
Intake Manifold Air Pressure
LV Battery Votlage
Purge Mode
Purge Duty Cycle
Purge Air Flow
Mass Air Flow
Target Idle Speed
Engine Speed
Intake Manifold Air Temperature
Fuel Level Percent
Fuel Tank Vapor Volume
Oil Pressure
A

In [ ]:
print("\n".join(m.find("Description").text for m in network_messages))

Components Miscelaneous 1
Torque Converter Slip
State of Torque Converter Clutch
RPMs and Temps Data
ActualCrankshaft Torque
Lateral and Longitudinal Accelerations
External Temperature
Break Data and others
ABS ESC and Brake Data
Wheelspeed Data
Bake Pedal Position
Transfer Case Range Status
Engine Gas Pedal and Others
Atmospheric Pressure and Thermal Management
Engine Oil Temperature
Terrain Mode and Engine Water Temperature
Torque Sums Max Mins
Electrical Machine Torque Data
AWD System Status
Current and Target Gear
Slope
Steering Angle
Engine Oil Temp etc MODE 21
Accum Air Port Mass Flow
Barometric Pressure
Engine Coolant Temperature
Shut Down Engine Coolant Temperature
Intake Air Temperature
Ambient Temperature
CAT Modeled Temperature
RPM vs Vehicle Speed Ratio
Intake Manifold Air Pressure
LV Battery Votlage
Purge Mode
Purge Duty Cycle
Purge Air Flow
Mass Air Flow
Target Idle Speed
Engine Speed
Intake Manifold Air Temperature
Fuel Level Percent
Fuel Tank Vapor Volume
Oil Pressure
A

In [ ]:
ecus_by_network['HS CAN'][0].find("GMLANDIDs").findall("GMLANDID")

[<Element 'GMLANDID' at 0x0000019FD5BC5580>,
 <Element 'GMLANDID' at 0x0000019FD5BC5760>,
 <Element 'GMLANDID' at 0x0000019FD5BC6200>,
 <Element 'GMLANDID' at 0x0000019FD5BC6980>,
 <Element 'GMLANDID' at 0x0000019FD5BC7D30>,
 <Element 'GMLANDID' at 0x0000019FD5BCC4A0>,
 <Element 'GMLANDID' at 0x0000019FD5BCD210>,
 <Element 'GMLANDID' at 0x0000019FD5BCDEE0>,
 <Element 'GMLANDID' at 0x0000019FD5BDD030>,
 <Element 'GMLANDID' at 0x0000019FD5D09EE0>,
 <Element 'GMLANDID' at 0x0000019FD5D0ABB0>,
 <Element 'GMLANDID' at 0x0000019FD5D0B470>,
 <Element 'GMLANDID' at 0x0000019FD5D0BC40>,
 <Element 'GMLANDID' at 0x0000019FD5D1A0C0>,
 <Element 'GMLANDID' at 0x0000019FD5D1ABB0>,
 <Element 'GMLANDID' at 0x0000019FD5D1B510>,
 <Element 'GMLANDID' at 0x0000019FD5D28040>,
 <Element 'GMLANDID' at 0x0000019FD5D2B9C0>,
 <Element 'GMLANDID' at 0x0000019FD5D3AC50>,
 <Element 'GMLANDID' at 0x0000019FD5D3BF10>]

In [ ]:
# Networks -> Modules -> Messages -> Signals

In [ ]:
int('302',16)

770